In [ ]:
# @title ##### License { display-mode: "form" }
# Copyright 2019 DeepMind Technologies Ltd. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Policy Distillation from AlphaZero

* This Colab trains an AlphaZero agent on Tic-Tac-Toe, distills it into a smaller network, and measures remaining playing strength.

## Install

In [ ]:
%pip install --upgrade "open_spiel>=2.0.1"
%pip install --upgrade "jax[cpu]" flax optax orbax-checkpoint chex matplotlib

In [ ]:
import collections
import os
import tempfile
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from flax import nnx

import pyspiel
from open_spiel.python.algorithms import mcts
from open_spiel.python.algorithms import minimax
from open_spiel.python.algorithms.alpha_zero import alpha_zero as az
from open_spiel.python.algorithms.alpha_zero import evaluator as evaluator_lib
from open_spiel.python.algorithms.alpha_zero import model_nnx as model_lib
from open_spiel.python.algorithms.alpha_zero import utils as az_utils

# Accelerator selection. Set USE_ACCELERATOR = False to pin everything to CPU.
# NOTE: self-play always runs on CPU regardless of this setting - az.alpha_zero()
# forks worker processes and an accelerator cannot be shared across forks.
USE_ACCELERATOR = True
if not USE_ACCELERATOR:
  jax.config.update("jax_platforms", "cpu")

print("jax", jax.__version__)
print("backend:", jax.default_backend())
print("devices:", jax.devices())
print("pyspiel games available:", len(pyspiel.registered_names()))

## Experiment Configuration

In [ ]:
# Hyperparameters and environment setup.
SEED = 42
GAME_NAME = "tic_tac_toe"

# Teacher parameters (AlphaZero)
TEACHER_WIDTH, TEACHER_DEPTH = 256, 4
TEACHER_TRAIN_STEPS = 20
TEACHER_ACTORS = 4
UCT_C = 1.41
SELFPLAY_SIMULATIONS = 40     # MCTS sims/move during self-play training.
EVAL_SIMULATIONS = 200        # MCTS sims/move when evaluating teacher+MCTS.
                              # 40 is on a cliff edge here: equally-converged
                              # teachers score 0 or 5+ losses vs minimax at 40,
                              # but both are perfect at 200.

# Student parameters (distilled model)
STUDENT_WIDTH, STUDENT_DEPTH = 32, 1
DISTILL_EPOCHS = 300
DISTILL_BATCH_SIZE = 128
DISTILL_LEARNING_RATE = 1e-2

# Evaluation parameters
N_GAMES = 100

game = pyspiel.load_game(GAME_NAME)
OBS_SHAPE = game.observation_tensor_shape()
NUM_ACTIONS = game.num_distinct_actions()
print(f"{GAME_NAME}: observation shape {OBS_SHAPE}, {NUM_ACTIONS} distinct actions")

# Train the AlphaZero Teacher

In [ ]:
# Train the teacher model using AlphaZero self-play.
teacher_dir = tempfile.mkdtemp(prefix="az_teacher_")

teacher_config = az.Config(
    game=GAME_NAME,
    path=teacher_dir,
    learning_rate=1e-3,
    weight_decay=1e-4,
    decouple_weight_decay=False,
    train_batch_size=2**7,
    replay_buffer_size=2**13,
    replay_buffer_reuse=4,
    max_steps=TEACHER_TRAIN_STEPS,
    checkpoint_freq=TEACHER_TRAIN_STEPS,
    actors=TEACHER_ACTORS,
    evaluators=1,
    evaluation_window=25,
    eval_levels=2,
    uct_c=UCT_C,
    max_simulations=SELFPLAY_SIMULATIONS,
    policy_alpha=1.0,
    policy_epsilon=0.25,
    temperature=1,
    temperature_drop=2,
    nn_model="mlp",
    nn_width=TEACHER_WIDTH,
    nn_depth=TEACHER_DEPTH,
    observation_shape=None,
    output_size=None,
    verbose=False,
    quiet=True,
    nn_api_version="nnx",
)

# Pin self-play to CPU: the forked actor processes cannot share an accelerator.
_prev_platforms = os.environ.get("JAX_PLATFORMS")
os.environ["JAX_PLATFORMS"] = "cpu"
_t0 = time.time()
try:
  az.alpha_zero(teacher_config)
finally:
  if _prev_platforms is None:
    os.environ.pop("JAX_PLATFORMS", None)
  else:
    os.environ["JAX_PLATFORMS"] = _prev_platforms
print(f"\nTeacher training finished in {(time.time() - _t0) / 60:.1f} minutes")
print("checkpoints:", sorted(f for f in os.listdir(teacher_dir) if f.startswith("checkpoint")))

## Reload the Teacher Model

In [ ]:
def build_model(width, depth, path, learning_rate=1e-3, seed=SEED):
  """Builds an AlphaZero Model (mlp torso) with the given capacity."""
  return model_lib.Model.build_model(
      model_type="mlp",
      input_shape=OBS_SHAPE,
      output_size=NUM_ACTIONS,
      nn_width=width,
      nn_depth=depth,
      weight_decay=1e-4,
      learning_rate=learning_rate,
      path=path,
      seed=seed,
  )


# Rebuild teacher architecture and load saved checkpoint.
teacher = build_model(TEACHER_WIDTH, TEACHER_DEPTH, teacher_dir)
teacher.load_checkpoint(TEACHER_TRAIN_STEPS)

TEACHER_PARAMS = int(teacher.num_trainable_variables)
print(f"Teacher reloaded: {TEACHER_PARAMS:,} trainable parameters "
      f"(mlp, width={TEACHER_WIDTH}, depth={TEACHER_DEPTH})")

## Batched Forward Pass

In [ ]:
def make_batched_forward(model):
  """Returns a batched forward function matching Model.inference."""
  net = nnx.merge(model._state.graphdef, model._state.params, model._state.batch_stats)  # pylint: disable=protected-access
  net.eval()
  graphdef, state = nnx.split(net)

  @jax.jit
  def _forward(state, obs, mask):
    net = nnx.merge(graphdef, state)
    logits, values = jax.vmap(net)(obs)
    logits = jnp.where(mask, logits, jnp.finfo(jnp.float32).min)
    return values, jax.nn.softmax(logits)

  return lambda obs, mask: _forward(
      state, jnp.asarray(obs, jnp.float32), jnp.asarray(mask, bool))


def state_features(state):
  """Returns observation tensor and legal actions mask for a state."""
  return (np.asarray(state.observation_tensor(), dtype=np.float32),
          np.asarray(state.legal_actions_mask(), dtype=bool))


# Verify that batched forward pass matches Model.inference().
_teacher_forward = make_batched_forward(teacher)
_probe = [game.new_initial_state()] + [game.new_initial_state().child(a)
                                       for a in game.new_initial_state().legal_actions()]
_obs = np.stack([state_features(s)[0] for s in _probe])
_mask = np.stack([state_features(s)[1] for s in _probe])
_vb, _pb = _teacher_forward(_obs, _mask)
for _i, _s in enumerate(_probe):
  _v, _p = teacher.inference(_obs[_i], _mask[_i])
  assert np.allclose(np.asarray(_p), np.asarray(_pb[_i]), atol=1e-5)
  assert np.allclose(np.asarray(_v), np.asarray(_vb[_i]), atol=1e-5)
print(f"Batched forward matches Model.inference() on {len(_probe)} probe states.")

# Evaluate the Teacher

In [ ]:
# Exact minimax solver cache.
_minimax_cache = {}


def optimal_value(state, player):
  """Returns game-theoretic value of state for player under optimal play."""
  key = (str(state), player)
  if key not in _minimax_cache:
    if state.is_terminal():
      _minimax_cache[key] = state.returns()[player]
    else:
      _minimax_cache[key] = minimax.alpha_beta_search(
          game, state=state, maximizing_player_id=player)[0]
  return _minimax_cache[key]


def optimal_actions(state):
  """Returns all value-preserving actions for current player."""
  player = state.current_player()
  best = optimal_value(state, player)
  return [a for a in state.legal_actions()
          if optimal_value(state.child(a), player) == best]


# Agent wrappers.
def make_random_agent(seed):
  rng = np.random.default_rng(seed)
  return lambda s: int(rng.choice(s.legal_actions()))


def make_optimal_agent(seed):
  rng = np.random.default_rng(seed)
  return lambda s: int(rng.choice(optimal_actions(s)))


def make_raw_agent(forward_fn):
  """Greedy policy head agent without MCTS search."""
  def act(state):
    obs, mask = state_features(state)
    _, policy = forward_fn(obs[None], mask[None])
    return int(np.argmax(np.asarray(policy[0])))
  return act


def make_mcts_agent(model, seed, simulations=EVAL_SIMULATIONS):
  """AlphaZero agent with MCTS search."""
  bot = mcts.MCTSBot(
      game,
      uct_c=UCT_C,
      max_simulations=simulations,
      evaluator=evaluator_lib.AlphaZeroEvaluator(game, model),
      solve=True,
      random_state=np.random.RandomState(seed),
  )
  return lambda s: int(bot.step(s))

In [ ]:
def play_matches(agent_a, agent_b, n_games):
  """Plays n_games alternating seats and returns (wins, draws, losses) for agent_a."""
  wins = draws = losses = 0
  for g in range(n_games):
    seat_a = g % 2
    state = game.new_initial_state()
    while not state.is_terminal():
      actor = agent_a if state.current_player() == seat_a else agent_b
      state.apply_action(actor(state))
    reward = state.returns()[seat_a]
    wins += reward > 0
    draws += reward == 0
    losses += reward < 0
  return wins, draws, losses


def format_wdl(record, n_games):
  w, d, l = record
  return f"{w:>3}W {d:>3}D {l:>3}L   (score {(w + 0.5 * d) / n_games:.3f})"


# Verify game evaluation harness (optimal vs optimal must draw).
_check = play_matches(make_optimal_agent(0), make_optimal_agent(1), 20)
print("sanity - optimal vs optimal:", format_wdl(_check, 20))
assert _check[0] == 0 and _check[2] == 0, "perfect play must always draw"

In [ ]:
# Evaluate teacher variants against random and minimax opponents.
random_agent = make_random_agent(SEED)
optimal_agent = make_optimal_agent(SEED)
teacher_raw_agent = make_raw_agent(_teacher_forward)
teacher_mcts_agent = make_mcts_agent(teacher, SEED)

teacher_results = {}
for name, agent in [("teacher-raw", teacher_raw_agent),
                    ("teacher+MCTS", teacher_mcts_agent)]:
  for opp_name, opp in [("random", make_random_agent(SEED + 1)),
                        ("minimax", make_optimal_agent(SEED + 1))]:
    t0 = time.time()
    teacher_results[(name, opp_name)] = play_matches(agent, opp, N_GAMES)
    print(f"{name:>13s} vs {opp_name:<8s} {format_wdl(teacher_results[(name, opp_name)], N_GAMES)}"
          f"   [{time.time() - t0:.1f}s]")

Search provides additional playing strength over the raw network outputs.

# Enumerate State Space

In [ ]:
def enumerate_non_terminal_states():
  """BFS from initial state to return deduplicated reachable non-terminal states."""
  root = game.new_initial_state()
  seen = {str(root): root}
  ordered = [root]
  queue = collections.deque([root])
  while queue:
    state = queue.popleft()
    for action in state.legal_actions():
      child = state.child(action)
      if child.is_terminal():
        continue
      key = str(child)
      if key not in seen:
        seen[key] = child
        ordered.append(child)
        queue.append(child)
  return ordered


all_states = enumerate_non_terminal_states()
NUM_STATES = len(all_states)
print(f"Reachable non-terminal positions: {NUM_STATES}")

## Symmetry-Aware Train/Validation Split

In [ ]:
# Compute board symmetries for Tic-Tac-Toe.
_grid = np.arange(9).reshape(3, 3)
SYMMETRIES = []
for _k in range(4):
  _rot = np.rot90(_grid, _k)
  for _variant in (_rot, np.fliplr(_rot)):
    _perm = tuple(_variant.reshape(-1).tolist())
    if _perm not in SYMMETRIES:
      SYMMETRIES.append(_perm)
assert len(SYMMETRIES) == 8, len(SYMMETRIES)


def board_cells(state):
  """Returns 9 board cells as a flat string."""
  return "".join(str(state).split())


def canonical_form(state):
  """Returns lexicographically smallest board over all 8 symmetries."""
  cells = board_cells(state)
  return min("".join(cells[perm[i]] for i in range(9)) for perm in SYMMETRIES)


symmetry_classes = collections.defaultdict(list)
for _idx, _state in enumerate(all_states):
  symmetry_classes[canonical_form(_state)].append(_idx)

class_keys = sorted(symmetry_classes)
_sizes = collections.Counter(len(v) for v in symmetry_classes.values())
print(f"{NUM_STATES} positions -> {len(class_keys)} symmetry classes")
print("class-size histogram:", dict(sorted(_sizes.items())))
assert sum(len(v) for v in symmetry_classes.values()) == NUM_STATES

In [ ]:
# Split symmetry classes into training and validation sets.
_rng = np.random.default_rng(SEED)
_shuffled = list(class_keys)
_rng.shuffle(_shuffled)
_n_val_classes = int(0.2 * len(_shuffled))

val_indices = np.array(sorted(
    i for k in _shuffled[:_n_val_classes] for i in symmetry_classes[k]))
train_indices = np.array(sorted(
    i for k in _shuffled[_n_val_classes:] for i in symmetry_classes[k]))

print(f"train: {len(train_indices)} positions from {len(_shuffled) - _n_val_classes} classes")
print(f"val:   {len(val_indices)} positions from {_n_val_classes} classes")

# Verify symmetry classes do not leak between train and validation splits.
_train_forms = {canonical_form(all_states[i]) for i in train_indices}
_val_forms = {canonical_form(all_states[i]) for i in val_indices}
assert not (_train_forms & _val_forms), "symmetry class present on both sides of the split"
assert len(set(train_indices) & set(val_indices)) == 0
print("No symmetry class appears on both sides of the split.")

## Generate Teacher Targets

In [ ]:
# Compute teacher target outputs for all states.
observations = np.stack([state_features(s)[0] for s in all_states])
legals_masks = np.stack([state_features(s)[1] for s in all_states])

_t0 = time.time()
_values, _policies = _teacher_forward(observations, legals_masks)
teacher_values = np.asarray(_values)
teacher_policies = np.asarray(_policies)
print(f"Teacher targets for {NUM_STATES} positions in {time.time() - _t0:.2f}s")

assert np.allclose(teacher_policies.sum(axis=1), 1.0, atol=1e-5)
assert np.all(teacher_policies[~legals_masks] < 1e-6)
print(f"policy targets: shape {teacher_policies.shape}, rows sum to 1")
print(f"value targets:  shape {teacher_values.shape}, "
      f"range [{teacher_values.min():.3f}, {teacher_values.max():.3f}]")

teacher_policy_entropy = float(np.mean(-np.sum(
    teacher_policies * np.log(np.clip(teacher_policies, 1e-12, None)), axis=1)))
print(f"teacher policy entropy: {teacher_policy_entropy:.4f} "
      f"(irreducible floor of the distillation cross-entropy)")

# Distill into Student Model

In [ ]:
# Build student model.
student_dir = tempfile.mkdtemp(prefix="az_student_")
student = build_model(STUDENT_WIDTH, STUDENT_DEPTH, student_dir,
                      learning_rate=DISTILL_LEARNING_RATE)
STUDENT_PARAMS = int(student.num_trainable_variables)
print(f"Student: {STUDENT_PARAMS:,} parameters "
      f"(mlp, width={STUDENT_WIDTH}, depth={STUDENT_DEPTH})")
print(f"Teacher: {TEACHER_PARAMS:,} parameters")
print(f"Compression: {TEACHER_PARAMS / STUDENT_PARAMS:.1f}x fewer parameters")

In [ ]:
def evaluate_losses(forward_fn, indices):
  """Computes policy cross-entropy and value loss against teacher targets."""
  values, policies = forward_fn(observations[indices], legals_masks[indices])
  policies = np.asarray(policies)
  values = np.asarray(values)
  targets = teacher_policies[indices]
  policy_loss = float(np.mean(-np.sum(
      targets * np.log(np.clip(policies, 1e-12, None)), axis=1)))
  value_loss = float(np.mean(0.5 * (values - teacher_values[indices]) ** 2))
  return policy_loss, value_loss


# Train the student model on teacher targets.
rng = np.random.default_rng(SEED)
history = {"train_policy": [], "train_value": [], "val_policy": [], "val_value": []}

_t0 = time.time()
for epoch in range(DISTILL_EPOCHS):
  order = rng.permutation(train_indices)
  epoch_policy, epoch_value, n_batches = 0.0, 0.0, 0

  for start in range(0, len(order), DISTILL_BATCH_SIZE):
    batch_idx = order[start:start + DISTILL_BATCH_SIZE]
    batch = az_utils.TrainInput(
        observation=jnp.asarray(observations[batch_idx]),
        legals_mask=jnp.asarray(legals_masks[batch_idx]),
        policy=jnp.asarray(teacher_policies[batch_idx]),
        value=jnp.asarray(teacher_values[batch_idx]),
    )
    losses = student.update(batch)
    epoch_policy += float(losses.policy)
    epoch_value += float(losses.value)
    n_batches += 1

  student_forward = make_batched_forward(student)
  val_policy, val_value = evaluate_losses(student_forward, val_indices)
  history["train_policy"].append(epoch_policy / n_batches)
  history["train_value"].append(epoch_value / n_batches)
  history["val_policy"].append(val_policy)
  history["val_value"].append(val_value)

  if epoch % 50 == 0 or epoch == DISTILL_EPOCHS - 1:
    print(f"epoch {epoch:>4d}  train policy {history['train_policy'][-1]:.4f} "
          f"value {history['train_value'][-1]:.4f}  |  "
          f"val policy {val_policy:.4f} value {val_value:.4f}")

print(f"\nDistillation finished in {time.time() - _t0:.1f}s")
student_forward = make_batched_forward(student)

In [ ]:
# Plot distillation loss curves.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, key, title in [(axes[0], "policy", "Policy loss (soft cross-entropy)"),
                       (axes[1], "value", "Value loss (0.5 x squared error)")]:
  ax.plot(history[f"train_{key}"], label="train")
  ax.plot(history[f"val_{key}"], label="validation")
  ax.set_xlabel("epoch")
  ax.set_ylabel("loss")
  ax.set_title(title)
  ax.legend()
  ax.grid(alpha=0.3)
fig.suptitle("Policy Distillation Loss")
fig.tight_layout()
plt.show()

# Evaluate the Student Model

In [ ]:
def exact_regret(action_fn, states=all_states):
  """Computes fraction of positions where agent move worsens optimal outcome."""
  mistakes = 0
  for state in states:
    player = state.current_player()
    best = optimal_value(state, player)
    achieved = optimal_value(state.child(action_fn(state)), player)
    if achieved < best:
      mistakes += 1
  return mistakes / len(states)


def greedy_actions(forward_fn):
  """Returns greedy action for every position in a batched pass."""
  _, policies = forward_fn(observations, legals_masks)
  return np.argmax(np.asarray(policies), axis=1)


# Verify exact regret metric.
print("regret of exact minimax  :", f"{exact_regret(lambda s: optimal_actions(s)[0]):.4f}",
      "(must be 0.0)")
print("regret of uniform random :",
      f"{exact_regret(make_random_agent(SEED)):.4f}")

In [ ]:
# Compute agreement and exact regret for student model.
student_raw_agent = make_raw_agent(student_forward)

teacher_greedy = greedy_actions(_teacher_forward)
student_greedy = greedy_actions(student_forward)

agreement_all = float(np.mean(teacher_greedy == student_greedy))
agreement_val = float(np.mean(teacher_greedy[val_indices] == student_greedy[val_indices]))

student_regret = exact_regret(student_raw_agent)
teacher_raw_regret = exact_regret(teacher_raw_agent)
teacher_mcts_regret = exact_regret(teacher_mcts_agent)
random_regret = exact_regret(make_random_agent(SEED))

print(f"Greedy agreement with teacher (all positions)  : {agreement_all:.3f}")
print(f"Greedy agreement (held-out validation set)    : {agreement_val:.3f}")
print()
print(f"Exact regret (student)     : {student_regret:.4f}")
print(f"Exact regret (teacher-raw) : {teacher_raw_regret:.4f}")
print(f"Exact regret (teacher+MCTS): {teacher_mcts_regret:.4f}")
print(f"Exact regret (random)      : {random_regret:.4f}")

In [ ]:
# Evaluate student model in matches against random and minimax.
student_results = {}
for opp_name, opp_seed in [("random", SEED + 1), ("minimax", SEED + 1)]:
  opp = make_random_agent(opp_seed) if opp_name == "random" else make_optimal_agent(opp_seed)
  student_results[opp_name] = play_matches(student_raw_agent, opp, N_GAMES)
  print(f"Student vs {opp_name:<8s} {format_wdl(student_results[opp_name], N_GAMES)}")

# Results Comparison

In [ ]:
def measure_latency(model, repeats=200):
  """Returns (Model.inference ms/state, jitted forward ms/state)."""
  probe = game.new_initial_state().child(4)
  obs, mask = state_features(probe)

  for _ in range(5):
    model.inference(obs, mask)
  t0 = time.time()
  for _ in range(repeats):
    model.inference(obs, mask)
  api_ms = (time.time() - t0) / repeats * 1e3

  forward = make_batched_forward(model)
  forward(obs[None], mask[None])[0].block_until_ready()
  t0 = time.time()
  for _ in range(repeats):
    forward(obs[None], mask[None])[0].block_until_ready()
  jit_ms = (time.time() - t0) / repeats * 1e3
  return api_ms, jit_ms


teacher_api_ms, teacher_jit_ms = measure_latency(teacher)
student_api_ms, student_jit_ms = measure_latency(student)
print(f"teacher: inference() {teacher_api_ms:.2f} ms | jitted {teacher_jit_ms:.4f} ms")
print(f"student: inference() {student_api_ms:.2f} ms | jitted {student_jit_ms:.4f} ms")

In [ ]:
def score(record):
  w, d, _ = record
  return (w + 0.5 * d) / N_GAMES


# Print summary table comparing parameters, speed, match win rates, and regret.
rows = [
    ("random baseline", None, None, None,
     score(play_matches(make_random_agent(SEED), make_random_agent(SEED + 3), N_GAMES)),
     score(play_matches(make_random_agent(SEED), make_optimal_agent(SEED + 2), N_GAMES)),
     random_regret),
    ("student (distilled)", STUDENT_PARAMS, student_api_ms, student_jit_ms,
     score(student_results["random"]), score(student_results["minimax"]), student_regret),
    ("teacher-raw", TEACHER_PARAMS, teacher_api_ms, teacher_jit_ms,
     score(teacher_results[("teacher-raw", "random")]),
     score(teacher_results[("teacher-raw", "minimax")]), teacher_raw_regret),
    ("teacher+MCTS", TEACHER_PARAMS, None, None,
     score(teacher_results[("teacher+MCTS", "random")]),
     score(teacher_results[("teacher+MCTS", "minimax")]), teacher_mcts_regret),
]

header = (f"{'agent':<22}{'params':>10}{'infer ms':>10}{'jit ms':>9}"
          f"{'vs random':>11}{'vs minimax':>12}{'exact regret':>14}")
print(header)
print("-" * len(header))
for name, params, api_ms, jit_ms, vs_rand, vs_mm, regret in rows:
  print(f"{name:<22}"
        f"{('-' if params is None else f'{params:,}'):>10}"
        f"{('-' if api_ms is None else f'{api_ms:.2f}'):>10}"
        f"{('-' if jit_ms is None else f'{jit_ms:.4f}'):>9}"
        f"{vs_rand:>11.3f}{vs_mm:>12.3f}"
        f"{('n/a' if regret is None else f'{regret:.4f}'):>14}")